# Primary GDSC/COSMIC drug-response analysis

**Purpose.** This is the complete locked computational experiment: source data, preprocessing, model development, final held-out evaluation, and feature interpretation.

**Assumes.** Raw GDSC/COSMIC files are under `../data/` and the project package is installed.

**Produces.** Reproducible cohort, model, test, and interpretation tables. Detailed evidence review is in [02_biological_context.ipynb](02_biological_context.ipynb).


## Research Question

Can anticancer drug response be predicted from COSMIC gene-expression features using GDSC models, and which features contribute to the locked model predictions? AUC is the primary target. Coefficient signs describe predictive association, not biological causation.


## Background and Data

GDSC release 8.4 supplies GDSC1/GDSC2 fitted single-agent responses and cell-line metadata. COSMIC Cell Lines Project v104 (GRCh38) supplies expression Z-scores. GDSC `COSMIC_ID` is linked to COSMIC samples by the validated sample-name relationship. Both AUC and LN_IC50 are retained at ingestion.


## Data Ingestion

Raw source CSV/TSV/XLSX files remain in `data/raw/`. The roughly 17.5-million-row COSMIC TSV has duplicate sample/gene measurements. Cache construction aggregates documented duplicates and writes long-format Parquet to `data/processed/cosmic_expression.parquet`. Temporary SQLite is used only during chunked aggregation. Selecting the cohort before querying this store avoids the previous approximately 72.8 GiB all-responses-by-all-genes allocation failure.


## Preprocessing

Reusable functions select the cohort before retrieving expression, retain one response per eligible cell line for a selected drug and screen, and keep metadata separate from features. Missingness filtering is target-independent; imputation and variance filtering are fitted on training data only. COSMIC inputs are already Z-scores, so scaling is disabled.


In [1]:
from gdsc.data import load_gdsc

gdsc = load_gdsc(
    data_dir="../data/raw",
    include_metadata=True,
)

gdsc.head()

/home/ajharris/Projects/gdsc-project/venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,DATASET,NLME_RESULT_ID,NLME_CURVE_ID,COSMIC_ID,CELL_LINE_NAME,SANGER_MODEL_ID,TCGA_DESC,DRUG_ID,DRUG_NAME,PUTATIVE_TARGET,...,Copy Number Alterations (CNA),Gene Expression,Methylation,Drug\nResponse,TISSUE_OF_ORIGIN,TISSUE_DESCRIPTOR_2,CANCER_TYPE,Microsatellite \ninstability Status (MSI),Screen Medium,Growth Properties
0,GDSC1,361,17635802,684057,ES5,SIDM00263,UNCLASSIFIED,1,Erlotinib,EGFR,...,Y,Y,Y,Y,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Adherent
1,GDSC1,361,17636176,684059,ES7,SIDM00269,UNCLASSIFIED,1,Erlotinib,EGFR,...,Y,Y,Y,Y,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Adherent
2,GDSC1,361,17636568,684062,EW-11,SIDM00203,UNCLASSIFIED,1,Erlotinib,EGFR,...,Y,Y,Y,Y,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Adherent
3,GDSC1,361,17636912,684072,SK-ES-1,SIDM01111,UNCLASSIFIED,1,Erlotinib,EGFR,...,Y,Y,Y,Y,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Semi-Adherent
4,GDSC1,361,17637300,687448,COLO-829,SIDM00909,SKCM,1,Erlotinib,EGFR,...,Y,Y,Y,Y,skin,melanoma,SKCM,MSS/MSI-L,R,Adherent


### Tissue selection

Coverage is summarized before choosing a cohort. Independent cell lines—not raw response rows—define the available sample size for a drug-specific model.


In [2]:
from gdsc.preprocessing import summarize_tissues

# Reusable summary; missing tissue labels are shown rather than silently removed.
tissue_summary = summarize_tissues(gdsc)
tissue_summary

,CELL_LINES,DRUGS,RESPONSE_OBSERVATIONS,cell_lines,observations
TISSUE_OF_ORIGIN,,,,,
lung_NSCLC,108,542,64499,108,64499
urogenital_system,104,542,61288,104,61288
leukemia,84,542,50008,84,50008
aero_dig_tract,77,542,45354,77,45354
lymphoma,69,542,40948,69,40948
lung_SCLC,63,542,33558,63,33558
skin,58,542,33253,58,33253
nervous_system,55,542,32794,55,32794
breast,52,542,31021,52,31021


### Selected tissue and drug eligibility

`lung_NSCLC` is selected because it has the largest coverage: 108 unique cell lines, 542 tested drugs, and 64,499 response observations. A drug is eligible at 75 unique cell lines. The threshold and deterministic availability rule are predefined, not chosen using performance.


In [3]:
import gdsc.preprocessing as preprocessing

coverage_report = preprocessing.analyze_cohort_drug_coverage(
    gdsc, tissue_of_origin="lung_NSCLC",
    thresholds=(20, 30, 40, 50, 60, 75, 90, 100),
)
drug_summary = coverage_report["drug_summary"]
identity_diagnostics = coverage_report["identity_diagnostics"]
duplicate_diagnostics = coverage_report["duplicate_diagnostics"]
coverage_diagnostics = coverage_report["coverage_diagnostics"]
eligibility_decision_table = coverage_report["eligibility_decision_table"]

print(coverage_diagnostics["distribution"])
display(eligibility_decision_table)
display(drug_summary.reset_index(drop=True).head(20))
print({key: len(value) for key, value in identity_diagnostics.items()})
{key: value for key, value in duplicate_diagnostics.items() if key != "duplicated_pairs"}


count    542.000000
mean      95.201107
std       21.824840
min        5.000000
25%       98.250000
50%      104.000000
75%      108.000000
max      108.000000
Name: N_CELL_LINES, dtype: float64


,minimum_unique_cell_lines,eligible_drugs
0,20,537
1,30,510
2,40,510
3,50,508
4,60,501
5,75,501
6,90,411
7,100,381


,DRUG_NAME,N_OBSERVATIONS,N_CELL_LINES,N_AUC_AVAILABLE,AUC_MISSING_FRACTION,N_LN_IC50_AVAILABLE,LN_IC50_MISSING_FRACTION,DATASETS,PUTATIVE_TARGET,DRUG_ID,observations,cell_lines
0,Selumetinib,394,108,394,0.0,394,0.0,"(GDSC1, GDSC2)","(MEK1, MEK2,)","(1062, 1498, 1736)",394,108
1,AZD4547,322,108,322,0.0,322,0.0,"(GDSC1, GDSC2)","(FGFR1, FGFR2, FGFR3, FGRF1, FGFR2, FGFR3)","(1135, 1497, 1786)",322,108
2,PLX-4720,321,108,321,0.0,321,0.0,"(GDSC1, GDSC2)","(BRAF,)","(1036, 1371)",321,108
3,AZD7762,320,108,320,0.0,320,0.0,"(GDSC1, GDSC2)","(CHEK1, CHEK2,)","(1022, 1402)",320,108
4,Olaparib,319,108,319,0.0,319,0.0,"(GDSC1, GDSC2)","(PARP1, PARP2,)","(1017, 1495)",319,108
5,Pictilisib,319,108,319,0.0,319,0.0,"(GDSC1, GDSC2)","(PI3K (class 1),)","(1058, 1527)",319,108
6,Afatinib,318,108,318,0.0,318,0.0,"(GDSC1, GDSC2)","(EGFR, ERBB2,)","(1032, 1377)",318,108
7,SN-38,318,108,318,0.0,318,0.0,"(GDSC1, GDSC2)","(TOP1,)","(1490, 1494)",318,108
8,Avagacestat,315,108,315,0.0,315,0.0,"(GDSC1, GDSC2)","(Amyloid beta20, Amyloid beta40,)","(205, 1072)",315,108
9,Gemcitabine,315,108,315,0.0,315,0.0,"(GDSC1, GDSC2)","(Pyrimidine antimetabolite,)","(135, 1190, 1393)",315,108


{'name_to_multiple_ids': 71, 'id_to_multiple_names': 0}


{'n_drug_cell_line_records': 64499,
 'n_unique_drug_cell_line_pairs': 57786,
 'n_duplicated_drug_cell_line_pairs': 6713,
 'max_records_per_pair': 2}

### Drug selection and response definition

The experiment selects the eligible drug with greatest cell-line coverage, breaking exact ties by lowest `DRUG_ID`. AUC is the primary target. GDSC1/GDSC2 are separate screens and are not averaged; the screen with greatest usable AUC coverage is selected.


In [4]:
experiment_response = preprocessing.build_initial_response_cohort(
    gdsc, tissue_of_origin="lung_NSCLC", min_unique_cell_lines=75, response_metric="AUC"
)
selected_drug = experiment_response["selected_drug"]
response_cohort = experiment_response["response_cohort"]
display(experiment_response["eligibility"]["eligible_drugs"].reset_index(drop=True).head(20))
display(selected_drug[["DRUG_ID", "DRUG_NAME", "N_CELL_LINES", "N_OBSERVATIONS", "DATASETS", "PUTATIVE_TARGET"]].to_frame().T)
display(experiment_response["dataset_coverage"])
print({
    "selected_dataset": experiment_response["selected_dataset"],
    "screen_specific_drug_id": experiment_response["selected_drug_id"],
    "response_rows": len(response_cohort),
    "unique_cell_lines": response_cohort["COSMIC_ID"].nunique(),
    "missing_auc_rows_excluded": experiment_response["n_excluded_response_rows"],
})


,DRUG_NAME,N_OBSERVATIONS,N_CELL_LINES,N_AUC_AVAILABLE,AUC_MISSING_FRACTION,N_LN_IC50_AVAILABLE,LN_IC50_MISSING_FRACTION,DATASETS,PUTATIVE_TARGET,DRUG_ID,observations,cell_lines
0,Selumetinib,394,108,394,0.0,394,0.0,"(GDSC1, GDSC2)","(MEK1, MEK2,)","(1062, 1498, 1736)",394,108
1,AZD4547,322,108,322,0.0,322,0.0,"(GDSC1, GDSC2)","(FGFR1, FGFR2, FGFR3, FGRF1, FGFR2, FGFR3)","(1135, 1497, 1786)",322,108
2,PLX-4720,321,108,321,0.0,321,0.0,"(GDSC1, GDSC2)","(BRAF,)","(1036, 1371)",321,108
3,AZD7762,320,108,320,0.0,320,0.0,"(GDSC1, GDSC2)","(CHEK1, CHEK2,)","(1022, 1402)",320,108
4,Olaparib,319,108,319,0.0,319,0.0,"(GDSC1, GDSC2)","(PARP1, PARP2,)","(1017, 1495)",319,108
5,Pictilisib,319,108,319,0.0,319,0.0,"(GDSC1, GDSC2)","(PI3K (class 1),)","(1058, 1527)",319,108
6,Afatinib,318,108,318,0.0,318,0.0,"(GDSC1, GDSC2)","(EGFR, ERBB2,)","(1032, 1377)",318,108
7,SN-38,318,108,318,0.0,318,0.0,"(GDSC1, GDSC2)","(TOP1,)","(1490, 1494)",318,108
8,Avagacestat,315,108,315,0.0,315,0.0,"(GDSC1, GDSC2)","(Amyloid beta20, Amyloid beta40,)","(205, 1072)",315,108
9,Gemcitabine,315,108,315,0.0,315,0.0,"(GDSC1, GDSC2)","(Pyrimidine antimetabolite,)","(135, 1190, 1393)",315,108


,DRUG_ID,DRUG_NAME,N_CELL_LINES,N_OBSERVATIONS,DATASETS,PUTATIVE_TARGET
97,1,Erlotinib,108,129,"(GDSC1, GDSC2)","(EGFR,)"


,DATASET,N_RESPONSE_ROWS,N_CELL_LINES,N_USABLE_RESPONSE_CELL_LINES,N_MISSING_RESPONSE
0,GDSC1,21,21,21,0
1,GDSC2,108,108,108,0


{'selected_dataset': 'GDSC2', 'screen_specific_drug_id': 1168, 'response_rows': 108, 'unique_cell_lines': 108, 'missing_auc_rows_excluded': 0}


### Build or reuse the expression feature store

This cache operation is safe to rerun: an existing processed cache is reused. It reports source rows processed and does not create a response-by-gene table.


In [5]:
from gdsc.cosmic import build_expression_cache

# Run once to create/reuse the preferred long-format feature store.
# It reads the raw COSMIC TSV in chunks, uses temporary SQLite aggregation,
# and writes data/processed/cosmic_expression.parquet (not a response-by-gene table).
processed_expression_cache = build_expression_cache(
    "../data", progress=print, total_source_rows=17_467_776
)
processed_expression_cache


Reusing existing expression cache: ../data/processed/cosmic_expression.parquet


PosixPath('../data/processed/cosmic_expression.parquet')

### Feature construction

After tissue, drug, screen, and response are fixed, the pipeline maps the retained cohort to COSMIC and retrieves only its expression. The result is the bounded cell-line-by-gene matrix; there is no outcome-driven feature selection.


In [6]:
experiment_dataset, experiment_response = preprocessing.build_initial_experiment_dataset(
    gdsc, data_dir="../data", tissue_of_origin="lung_NSCLC",
    min_unique_cell_lines=75, response_metric="AUC",
    max_gene_missing_fraction=0.20,
)
missingness = experiment_dataset.diagnostics["missingness_before_filtering"]
preprocessing_report = {
    "response_cell_lines": experiment_dataset.diagnostics["n_response_cell_lines"],
    "mapped_cosmic_samples": experiment_dataset.diagnostics["n_mapped_to_cosmic"],
    "unmatched_cell_lines": experiment_dataset.diagnostics["n_unmatched"],
    "expression_available_cell_lines": experiment_dataset.diagnostics["n_with_expression"],
    "excluded_no_expression": len(experiment_dataset.diagnostics["excluded_no_expression_ids"]),
    "genes_before_filtering": missingness["n_genes"],
    "genes_after_filtering": experiment_dataset.diagnostics["n_genes_after_filtering"],
    "all_missing_genes": len(missingness["all_missing_genes"]),
    "total_missing_values": missingness["total_missing"],
    "overall_missing_fraction": missingness["overall_missing_fraction"],
    "X_shape": experiment_dataset.X.shape,
    "y_shape": experiment_dataset.y.shape,
    "metadata_shape": experiment_dataset.metadata.shape,
}
assert len(experiment_dataset.X) == len(experiment_dataset.y) == len(experiment_dataset.metadata)
assert experiment_dataset.X.index.equals(experiment_dataset.y.index)
assert experiment_dataset.X.index.equals(experiment_dataset.metadata.index)
preprocessing_report


{'response_cell_lines': 108,
 'mapped_cosmic_samples': 108,
 'unmatched_cell_lines': 0,
 'expression_available_cell_lines': 106,
 'excluded_no_expression': 2,
 'genes_before_filtering': 16980,
 'genes_after_filtering': 16980,
 'all_missing_genes': 0,
 'total_missing_values': 0,
 'overall_missing_fraction': 0.0,
 'X_shape': (106, 16980),
 'y_shape': (106,),
 'metadata_shape': (106, 9)}

### Train, validation, and test split

Splits are grouped by `COSMIC_ID`, preventing a cell line from appearing in more than one partition. The test partition remains isolated throughout development.


In [7]:
splits = preprocessing.split_by_cell_line(
    experiment_dataset, test_fraction=0.20, validation_fraction=0.20, random_state=42
)
train, validation, test = splits["train"], splits["validation"], splits["test"]
fitted_preprocessor = preprocessing.build_preprocessor(imputation="median", scaling=False)
X_train = fitted_preprocessor.fit_transform(train.X)
X_val = fitted_preprocessor.transform(validation.X)
X_test = fitted_preprocessor.transform(test.X)
y_train, y_val, y_test = train.y, validation.y, test.y
metadata_train, metadata_val, metadata_test = train.metadata, validation.metadata, test.metadata
split_ids = [set(part.metadata["COSMIC_ID"]) for part in (train, validation, test)]
assert not (split_ids[0] & split_ids[1] or split_ids[0] & split_ids[2] or split_ids[1] & split_ids[2])
modeling_handoff = {
    "X_train": X_train.shape, "X_val": X_val.shape, "X_test": X_test.shape,
    "y_train": y_train.shape, "y_val": y_val.shape, "y_test": y_test.shape,
    "metadata_train": metadata_train.shape, "metadata_val": metadata_val.shape,
    "metadata_test": metadata_test.shape,
    "genes_after_training_only_variance_filter": X_train.shape[1],
    "scaling": "disabled: COSMIC values are already Z-scores",
}
modeling_handoff


{'X_train': (63, 16980),
 'X_val': (21, 16980),
 'X_test': (22, 16980),
 'y_train': (63,),
 'y_val': (21,),
 'y_test': (22,),
 'metadata_train': (63, 9),
 'metadata_val': (21, 9),
 'metadata_test': (22, 9),
 'genes_after_training_only_variance_filter': 16980,
 'scaling': 'disabled: COSMIC values are already Z-scores'}

## Modeling

The development sequence is a mean-response baseline, fixed regularized linear baselines, and a small training-only candidate comparison. Ridge and Elastic Net are high-dimensional baselines; the shallow Random Forest is a bounded nonlinear comparison. RMSE is the primary selection metric.


In [8]:
import pandas as pd
from gdsc.evaluation import fit_and_evaluate_validation
from gdsc.models import build_dummy_regressor, build_elastic_net_model, build_ridge_model

baseline_models = {
    "Mean baseline": build_dummy_regressor(),
    "Ridge (alpha=1.0)": build_ridge_model(alpha=1.0),
    "Elastic Net (alpha=1.0, l1_ratio=0.5)": build_elastic_net_model(),
}
validation_results = []
for name, model in baseline_models.items():
    _, metrics = fit_and_evaluate_validation(model, X_train, y_train, X_val, y_val)
    validation_results.append({"MODEL": name, **metrics.__dict__})

print({"train_cell_lines": len(y_train), "validation_cell_lines": len(y_val),
       "held_out_test_cell_lines": len(y_test), "genes": X_train.shape[1],
       "features_per_training_cell_line": X_train.shape[1] / len(y_train)})
display(pd.DataFrame(validation_results).set_index("MODEL"))
# X_test and y_test are deliberately not used in this baseline-comparison cell.


{'train_cell_lines': 63, 'validation_cell_lines': 21, 'held_out_test_cell_lines': 22, 'genes': 16980, 'features_per_training_cell_line': 269.5238095238095}


,mae,rmse,pearson,spearman,r2
MODEL,,,,,
Mean baseline,0.068166,0.088695,NaN,NaN,-0.010572
Ridge (alpha=1.0),0.066239,0.081292,0.445491,0.354545,0.151076
"Elastic Net (alpha=1.0, l1_ratio=0.5)",0.068166,0.088695,NaN,NaN,-0.010572


### Candidate model development

Candidates are tuned by three-fold cross-validation within training data, then evaluated once on validation. The held-out test set is excluded. The selected configuration is tuned Ridge with `alpha=100.0`.


In [9]:
from gdsc.models import (
    build_elastic_net_model, build_random_forest_model, build_ridge_model,
    tune_training_only,
)
from gdsc.evaluation import evaluate_regression

candidate_specs = {
    "Tuned Ridge": (build_ridge_model(), {"alpha": [0.01, 0.1, 1.0, 10.0, 100.0]}),
    "Tuned Elastic Net": (build_elastic_net_model(), {"alpha": [0.01, 0.1, 1.0], "l1_ratio": [0.1, 0.5, 0.9]}),
    # A single shallow, 50-tree configuration is a bounded nonlinear comparison.
    # A large forest grid is impractical with 16,980 genes and 63 training lines.
    "Random Forest (bounded)": (build_random_forest_model(n_estimators=50, max_depth=5, min_samples_leaf=2, max_features=0.1), {"max_depth": [5]}),
}
candidate_results = []
for name, (estimator, grid) in candidate_specs.items():
    fitted, cv_result = tune_training_only(estimator, grid, X_train, y_train, cv=3)
    metrics = evaluate_regression(y_val, fitted.predict(X_val))
    candidate_results.append({"MODEL": name, **cv_result, **metrics.__dict__})

validation_candidates = pd.DataFrame(candidate_results).set_index("MODEL").sort_values("rmse")
validation_candidates  # VALIDATION RESULTS ONLY: X_test/y_test are not used here.


,parameters,cv_rmse_mean,cv_rmse_std,mae,rmse,pearson,spearman,r2
MODEL,,,,,,,,
Tuned Ridge,{'alpha': 100.0},0.102637,0.010035,0.066219,0.081280,0.445106,0.354545,0.151336
Random Forest (bounded),{'max_depth': 5},0.109097,0.023194,0.065792,0.083055,0.358506,0.224675,0.113860
Tuned Elastic Net,"{'alpha': 1.0, 'l1_ratio': 0.5}",0.108449,0.029802,0.068166,0.088695,NaN,NaN,-0.010572


## Final Test Evaluation

**The model is locked before this cell runs.** Ridge with `alpha=100.0`, median imputation, and training-only variance filtering is fitted on 63 training cell lines and 16,980 genes. No post-test changes to preprocessing, features, hyperparameters, or model family are permitted.

The recorded locked-Ridge test result is MAE 0.050805, RMSE 0.060515, Pearson 0.354313, Spearman 0.247883, and R² -0.423419. The mean-response RMSE is 0.061048. These are fixed generalization results, not a basis for further tuning.


In [10]:
from gdsc.evaluation import evaluate_locked_model
from gdsc.models import build_dummy_regressor, build_ridge_model

# No tuning occurs here. Both estimators fit X_train/y_train only.
_, dummy_test_metrics, dummy_predictions = evaluate_locked_model(
    build_dummy_regressor(), X_train, y_train, X_test, y_test, metadata_test
)
_, ridge_test_metrics, ridge_predictions = evaluate_locked_model(
    build_ridge_model(alpha=100.0), X_train, y_train, X_test, y_test, metadata_test
)
held_out_test_results = pd.DataFrame([
    {"MODEL": "Mean-response baseline", **dummy_test_metrics.__dict__},
    {"MODEL": "Locked Ridge (alpha=100.0)", **ridge_test_metrics.__dict__},
]).set_index("MODEL")
held_out_test_results  # HELD-OUT TEST RESULTS: do not retune after this cell.


,mae,rmse,pearson,spearman,r2
MODEL,,,,,
Mean-response baseline,0.054069,0.061048,NaN,NaN,-0.448617
Locked Ridge (alpha=100.0),0.050805,0.060515,0.354313,0.247883,-0.423419


## Feature Interpretation

Signed Ridge coefficients rank contribution to locked predictions. Bootstrap refits use training data only and the same locked configuration; they assess stability without changing the model or test result.


In [11]:
from gdsc.interpretation import ridge_bootstrap_stability, ridge_coefficients
from gdsc.models import build_ridge_model
locked_ridge = build_ridge_model(alpha=100.0).fit(X_train, y_train)
feature_ranking = ridge_coefficients(locked_ridge, train.X.columns)
feature_stability = ridge_bootstrap_stability(lambda: build_ridge_model(alpha=100.0), X_train, y_train, train.X.columns, n_resamples=100, random_state=42)
feature_ranking.head(20).merge(feature_stability, on="GENE_SYMBOL")


,GENE_SYMBOL,COEFFICIENT,ABS_COEFFICIENT,SIGN,COEFFICIENT_MEAN,COEFFICIENT_MEDIAN,COEFFICIENT_STD,POSITIVE_FRACTION,NEGATIVE_FRACTION
0,YTHDF3,0.000285,0.000285,1,0.000190,0.000205,0.000067,1.00,0.00
1,BAG5,-0.000247,0.000247,-1,-0.000167,-0.000201,0.000080,0.03,0.97
2,CYB5B,0.000235,0.000235,1,0.000151,0.000199,0.000085,0.98,0.02
3,SETD3,0.000225,0.000225,1,0.000139,0.000180,0.000079,0.98,0.02
4,TMEM11,-0.000223,0.000223,-1,-0.000148,-0.000148,0.000042,0.00,1.00
5,WDR13,0.000221,0.000221,1,0.000135,0.000164,0.000078,0.92,0.08
6,MSTN,0.000220,0.000220,1,0.000139,0.000157,0.000057,0.98,0.02
7,GSTA3,-0.000220,0.000220,-1,-0.000146,-0.000148,0.000047,0.00,1.00
8,DHRS7B,-0.000216,0.000216,-1,-0.000148,-0.000149,0.000033,0.00,1.00
9,RTF1,-0.000215,0.000215,-1,-0.000130,-0.000145,0.000042,0.00,1.00


### Reading the feature results

Positive coefficients associate higher expression with higher predicted AUC; negative coefficients associate it with lower predicted AUC. `ABS_COEFFICIENT` determines rank within this fitted model, not biological importance. Resampling means, medians, standard deviations, and direction fractions describe stability. Ridge can distribute weight across correlated genes, so rankings are predictive associations rather than causal claims.


### Correlation among top features

These training-only correlations describe whether high-ranked genes may represent related expression patterns. They do not remove features or refit the model.


In [12]:
from gdsc.interpretation import top_feature_correlations
top_gene_names = feature_ranking.head(20)["GENE_SYMBOL"].tolist()
top_feature_correlation_pairs = top_feature_correlations(train.X, top_gene_names, top_n=20)
top_feature_correlation_pairs.head(20)


,GENE_A,GENE_B,CORRELATION,ABS_CORRELATION
0,TMEM11,DHRS7B,0.752273,0.752273
1,TMEM11,AKAP10,0.670153,0.670153
2,CYB5B,DDT,0.489348,0.489348
3,BAG5,ISCA2,0.489120,0.489120
4,DHRS7B,AKAP10,0.481228,0.481228
5,DDT,NDUFB9,0.481068,0.481068
6,DHRS7B,SRR,0.446882,0.446882
7,CYB5B,SFR1,0.434718,0.434718
8,SETD3,DDTL,0.384198,0.384198
9,BAG5,CYB5B,-0.367261,0.367261


## Biological Context Summary

Stable predictive gene associations merit comparison with existing biological evidence, while keeping model finding, literature evidence, and hypothesis distinct. Predictive importance is not causality. The detailed framework and evidence template are in [02_biological_context.ipynb](02_biological_context.ipynb).


## Limitations

This is a single-drug, single-tissue cell-line experiment with 106 expression-available lines and many more features than training samples. Validation/test partitions are small, response direction depends on the GDSC AUC definition, and correlated genes complicate individual coefficients. Results are predictive, not clinical or mechanistic claims.


## Conclusions

The project established a memory-safe GDSC/COSMIC workflow, selected Erlotinib from GDSC2 in `lung_NSCLC`, and locked a Ridge model after validation-only development. Its held-out result and feature ranking are fixed records for the AUC experiment.


## Next Steps

1. Complete targeted literature review in [02_biological_context.ipynb](02_biological_context.ipynb) without changing computational results.
2. Use [03_sensitivity_ln_ic50.ipynb](03_sensitivity_ln_ic50.ipynb) for the future, separate LN_IC50 sensitivity analysis.
